# Multi-Stage Queries in SLayer

Most semantic layers parameterize a single GROUP BY. But what if you need to aggregate *an aggregation* — like averaging monthly revenue across stores, or bucketing users by a computed metric?

SLayer handles this naturally: every query implies a model (via auto-introspection of its result columns), so you can chain queries together. This notebook demonstrates two multi-stage patterns with working code.

See also: [Query Lists](../../concepts/queries.md#query-lists) | [ModelExtension](../../concepts/queries.md#modelextension) | [Creating Models from Queries](../../concepts/models.md#creating-models-from-queries)

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

## Queries as Models

In most semantic layers, queries and models are entirely separate concepts. In SLayer, every query resolves to SQL, and SLayer's [auto-introspection](../../concepts/ingestion.md) can generate a model from any SQL SELECT query. Put these together: **every query automatically implies a model**.

When a query becomes a virtual model, its result columns become dimensions, and numeric columns also get auto-generated aggregation measures. This means you can use one query's output as the input to another — no special syntax needed.

## Example 1: Average Monthly Revenue per Store

**Goal:** What is the average monthly revenue for each store?

This is a two-stage query:
1. **Inner query:** Total revenue grouped by store and month
2. **Outer query:** Average the monthly revenue, grouped by store

The inner query's `order_total_sum` result column becomes a dimension on the virtual model, which we can then aggregate with `order_total_sum:avg` in the outer query.

In [2]:
# Stage 1: Monthly revenue by store
inner_result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["order_total:sum"],
        "dimensions": ["stores.name"],
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 10,
    }
)

print("Inner query result (first 10 rows):")
print(f"{'Month':<12} {'Store':<20} {'Revenue':>12}")
print("-" * 46)
for row in inner_result.data:
    month = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    print(f"{month:<12} {store:<20} ${rev:>11,.2f}")

Inner query result (first 10 rows):
Month        Store                     Revenue
----------------------------------------------
2023-05      Philadelphia         $  15,102.20
2023-06      Philadelphia         $  23,098.27
2023-07      Philadelphia         $  29,880.94
2023-08      Philadelphia         $  38,628.40
2023-09      Philadelphia         $  44,370.20
2023-10      Philadelphia         $  47,540.71
2023-11      Philadelphia         $  51,803.50
2023-11      Brooklyn             $  19,512.44
2023-12      Brooklyn             $  41,531.31
2023-12      Philadelphia         $  55,058.78


Now we chain the two queries together. We give the inner query a `name` and pass both queries as a list to `engine.execute_sync()`. The outer query references the inner query's name as its `source_model`:

In [3]:
# Two-stage query: average monthly revenue per store
result = engine.execute_sync(
    query=[
        {
            "name": "monthly_store_revenue",
            "source_model": "orders",
            "measures": ["order_total:sum"],
            "dimensions": ["stores.name"],
            "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        },
        {
            "source_model": "monthly_store_revenue",
            "measures": ["order_total_sum:avg"],
            "dimensions": ["stores__name"],
            "order": [{"column": "order_total_sum_avg", "direction": "desc"}],
        },
    ]
)

print("Average monthly revenue by store:")
print(f"{'Store':<20} {'Avg Monthly Revenue':>20}")
print("-" * 42)
for row in result.data:
    store = row["monthly_store_revenue.stores__name"]
    avg_rev = row["monthly_store_revenue.order_total_sum_avg"]
    print(f"{store:<20} ${avg_rev:>19,.2f}")

Average monthly revenue by store:
Store                 Avg Monthly Revenue
------------------------------------------
Brooklyn             $          90,118.03
Chicago              $          69,633.27
San Francisco        $          62,331.80
Philadelphia         $          56,737.63
New Orleans          $          25,826.10


**What happened under the hood:**

1. SLayer executed the inner query (`monthly_store_revenue`), producing rows of (store, month, revenue)
2. It auto-introspected those results to create a virtual model with:
   - Dimensions: `stores__name`, `ordered_at`, `order_total_sum`
   - Measures: one per numeric column (e.g., `order_total_sum`), queryable with aggregations like `order_total_sum:avg`, `order_total_sum:min`, etc. `*:count` is always available.
3. The outer query used this virtual model, grouping by `stores__name` and requesting `order_total_sum_avg`

The virtual model is a self-contained table — it no longer has the joins the source model had. That's why the dimension is called `stores__name` (using `__` to encode the original join path) rather than `stores.name` (which would imply a join to a `stores` model that doesn't exist on the virtual model).

No special multi-stage syntax was needed — just a named query and a reference to it.

In [4]:
print("Generated SQL:")  # NOSONAR(S1192) — tutorial cells are intentionally self-contained; pulling this label into a top-level constant would force readers to scroll between cells
print(result.sql)

Generated SQL:
SELECT
  monthly_store_revenue.stores__name AS "monthly_store_revenue.stores__name",
  AVG(monthly_store_revenue.order_total_sum) AS "monthly_store_revenue.order_total_sum_avg"
FROM (
  SELECT
    "orders.stores.name" AS stores__name,
    "orders.ordered_at" AS ordered_at,
    "orders.order_total_sum" AS order_total_sum
  FROM (
    SELECT
      stores.name AS "orders.stores.name",
      DATE_TRUNC('MONTH', orders.ordered_at) AS "orders.ordered_at",
      SUM(orders.order_total) AS "orders.order_total_sum"
    FROM orders AS orders
    LEFT JOIN stores AS stores
      ON orders.store_id = stores.id
    GROUP BY
      stores.name,
      DATE_TRUNC('MONTH', orders.ordered_at)
  ) AS _inner
) AS monthly_store_revenue
GROUP BY
  monthly_store_revenue.stores__name
ORDER BY
  "monthly_store_revenue.order_total_sum_avg" DESC


## Example 2: Orders by Customer Activity Bucket

**Goal:** How many orders and how much revenue comes from low, medium, and high-activity customers?

This is a two-stage pattern with a join:
1. **Inner query:** Compute total order count per customer
2. **Outer query:** Join the inner result to `orders`, add a CASE-based bucket dimension via [ModelExtension](../../concepts/queries.md#modelextension), and group by the bucket

The bucketing uses an [inline dimension](../../concepts/queries.md#modelextension) — a CASE expression that categorizes each customer's total order count into Low / Medium / High.

In [5]:
# Stage 1: Total orders per customer
inner_result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["*:count"],
        "dimensions": ["customer_id"],
        "order": [{"column": {"name": "_count"}, "direction": "desc"}],
        "limit": 10,
    }
)

print("Top 10 customers by order count:")
print(f"{'Customer ID':<40} {'Orders':>10}")
print("-" * 52)
for row in inner_result.data:
    cust = row["orders.customer_id"]
    count = row["orders._count"]
    print(f"{cust:<40} {count:>10,}")

Top 10 customers by order count:
Customer ID                                  Orders
----------------------------------------------------
8022565c-2f5c-4187-a949-9e7fb1b7f7c8            790
5a27aad0-84b5-4046-995e-d4688a4c3904            788
4ccf5702-bf9f-45c6-9730-1c203fdbe61a            787
e5a2886c-461f-4bbe-91a6-96e6cf647b0f            786
7f61f295-6a3b-4336-b388-f194f1518af6            780
8a3c04f6-6405-40a9-9a9a-37b6bf52049a            778
c26e532f-ff72-4ab8-9911-75908153d4f4            776
d2773711-f8b5-449d-ac51-fc8bb8b3ab1e            774
de190984-1e4e-4fe3-a1c8-4e6fd7924a72            773
b3b3860a-e95e-4735-bd60-338e6a89f38c            772


Now we chain the queries. The outer query uses `ModelExtension` to:
1. **Join** the inner query (customer order counts) to the `orders` model via `customer_id`
2. **Define an inline dimension** using a CASE expression that buckets customers by their total order count

Since the bucket dimension references a column from the joined sub-query, its SQL uses the joined table alias: `customer_activity._count` (the sub-query name followed by the column).

In [6]:
# Two-stage query: orders grouped by customer activity bucket
result = engine.execute_sync(
    query=[
        {
            "name": "customer_activity",
            "source_model": "orders",
            "measures": ["*:count"],
            "dimensions": ["customer_id"],
        },
        {
            "source_model": {
                "source_name": "orders",
                "joins": [
                    {
                        "target_model": "customer_activity",
                        "join_pairs": [["customer_id", "customer_id"]],
                    }
                ],
                "columns": [
                    {
                        "name": "activity_bucket",
                        "sql": "CASE WHEN customer_activity._count >= 500 THEN 'High' WHEN customer_activity._count >= 200 THEN 'Medium' ELSE 'Low' END",
                        "type": "string",
                    }
                ],
            },
            "measures": ["*:count", "order_total:sum"],
            "dimensions": ["activity_bucket"],
            "order": [{"column": "order_total_sum", "direction": "desc"}],
        },
    ]
)

print("Orders by customer activity bucket:")
print(f"{'Bucket':<12} {'Orders':>10} {'Revenue':>14}")
print("-" * 38)
for row in result.data:
    bucket = row["orders.activity_bucket"]
    count = row["orders._count"]
    rev = row["orders.order_total_sum"]
    print(f"{bucket:<12} {count:>10,} ${rev:>13,.2f}")

Orders by customer activity bucket:
Bucket           Orders        Revenue
--------------------------------------
Medium          330,132 $ 3,533,437.09
Low             132,298 $ 2,564,860.66
High            198,716 $ 1,218,842.55


In [7]:
print("Generated SQL:")
print(result.sql)

Generated SQL:
SELECT
  CASE
    WHEN customer_activity._count >= 500
    THEN 'High'
    WHEN customer_activity._count >= 200
    THEN 'Medium'
    ELSE 'Low'
  END AS "orders.activity_bucket",
  COUNT(*) AS "orders._count",
  SUM(orders.order_total) AS "orders.order_total_sum"
FROM orders AS orders
LEFT JOIN (
  (
    SELECT
      "orders.customer_id" AS customer_id,
      "orders._count" AS _count
    FROM (
      SELECT
        orders.customer_id AS "orders.customer_id",
        COUNT(*) AS "orders._count"
      FROM orders AS orders
      GROUP BY
        orders.customer_id
    ) AS _inner
  )
) AS customer_activity
  ON orders.customer_id = customer_activity.customer_id
GROUP BY
  CASE
    WHEN customer_activity._count >= 500
    THEN 'High'
    WHEN customer_activity._count >= 200
    THEN 'Medium'
    ELSE 'Low'
  END
ORDER BY
  "orders.order_total_sum" DESC


## Example 3: Parallel Siblings Joined in a Final Stage

Stages in a query list are not restricted to a linear pipeline. Any stage may use any *prior* named sibling stage as its `source_model` or as a `joins.target_model`, so several rollups can run in parallel and feed a single final stage.

Below: stage 1 aggregates orders per customer; stage 2 — itself a non-final named stage — joins that result back to `customers` to attach the per-customer total to each row; stage 3 takes the max.

In [8]:
# Three-stage DAG: kpis (per-customer rollup) → tagged (joined back to customers) → final (max)
result = engine.execute_sync(
    query=[
        {
            "name": "kpis",
            "source_model": "orders",
            "dimensions": ["customer_id"],
            "measures": ["order_total:sum"],
        },
        {
            "name": "tagged",
            "source_model": {
                "source_name": "customers",
                "joins": [
                    {
                        "target_model": "kpis",
                        "join_pairs": [["id", "customer_id"]],
                    }
                ],
            },
            "dimensions": ["name", "kpis.order_total_sum"],
        },
        {
            "source_model": "tagged",
            "measures": ["kpis__order_total_sum:max"],
        },
    ]
)

print("Max per-customer order total:")
for row in result.data:
    for k, v in row.items():
        print(f"  {k} = {v:,.2f}")

Max per-customer order total:
  tagged.kpis__order_total_sum_max = 13,357.66


**What's new vs. Example 2:** in Example 2 only the *final* stage joined a named sibling. Here, the **non-final** stage `tagged` joins the prior named stage `kpis`, so the structure is a DAG with three stages instead of a two-step chain.

Forward references and self references are rejected with a clear error — a stage may only resolve to stages defined earlier in the list.

In [9]:
print("Generated SQL:")
print(result.sql)

Generated SQL:
SELECT
  MAX(tagged.kpis__order_total_sum) AS "tagged.kpis__order_total_sum_max"
FROM (
  SELECT
    "customers.name" AS name,
    "customers.kpis.order_total_sum" AS kpis__order_total_sum
  FROM (
    SELECT
      customers.name AS "customers.name",
      kpis.order_total_sum AS "customers.kpis.order_total_sum"
    FROM customers AS customers
    LEFT JOIN (
      (
        SELECT
          "orders.customer_id" AS customer_id,
          "orders.order_total_sum" AS order_total_sum
        FROM (
          SELECT
            orders.customer_id AS "orders.customer_id",
            SUM(orders.order_total) AS "orders.order_total_sum"
          FROM orders AS orders
          GROUP BY
            orders.customer_id
        ) AS _inner
      )
    ) AS kpis
      ON customers.id = kpis.customer_id
  ) AS _inner
) AS tagged


## Summary

Multi-stage queries in SLayer require no special syntax — they emerge naturally from two features:

| Feature | Role in Multi-Stage Queries |
|---------|----------------------------|
| **Query-as-model** | Every query's result auto-generates a virtual model with dimensions and measures |
| **Query lists** | Pass `[inner, outer]` to `engine.execute_sync()` — inner queries are named, outer references them |
| **Auto-generated measures** | Numeric columns from inner queries get one measure each, queryable with any aggregation |
| **ModelExtension joins** | Join a sub-query's virtual model to any stored model at query time |
| **Inline dimensions** | Add CASE expressions or other computed dimensions via `ModelExtension` |

See the [Query Lists docs](../../concepts/queries.md#query-lists) and [ModelExtension docs](../../concepts/queries.md#modelextension) for the full reference.